# Axis — CODE-15 deep-backbone pretraining (cloud)

Pretrains `ECGResNetDeep1D` on CODE-15% (345k ECGs) and produces a small
`backbone.pt` (~27 MB) to bring back to the laptop for the local fine-tune.

**Before running:** `Runtime -> Change runtime type -> GPU` (T4 works; A100 is faster).
On free Colab the 50 GB corpus is tight — see the optional Drive cell.

In [ ]:
# 1) Confirm we have a GPU and enough disk
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h / | tail -1

In [ ]:
# 2) Get the code (public repo, ~16 MB)
!git clone https://github.com/Davmunrey/Heartcheck
%cd Heartcheck

In [ ]:
# 3) (OPTIONAL) If the 50 GB won't fit on the Colab disk, mount Drive and
#    point CODE15_ROOT there. Skip this cell if you have enough local disk.
# from google.colab import drive; drive.mount('/content/drive')
# %env CODE15_ROOT=/content/drive/MyDrive/code_15pct

In [ ]:
# 4) ONE COMMAND: install deps, download CODE-15, pretrain. (~2-4 h on A100)
#    Tunables: EPOCHS, BATCH, LR, WORKERS. Drop BATCH if you hit OOM.
!EPOCHS=10 BATCH=128 WORKERS=8 bash scripts/cloud_pretrain_code15.sh

In [ ]:
# 5) Download the backbone to your machine (only ~27 MB)
from google.colab import files
files.download('runs/cloud/code15_pretrain/backbone.pt')

## Back on the laptop
```bash
mkdir -p runs/local/code15_pretrain
mv ~/Downloads/backbone.pt runs/local/code15_pretrain/backbone.pt
PRETRAIN_OUT=runs/local/code15_pretrain scripts/pretrain_finetune_code15.sh
```
Then evaluate vs the champion's 0.608 and promote only if it wins. See
`docs/CLOUD_PRETRAIN.md`.